# GSM8K dataset preparation

Load the official GSM8K splits, normalize their records, and create a reproducible development subset from training data. No GPU is required.

In [ ]:
from pathlib import Path
import os

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    !git clone {REPOSITORY_URL} {REPOSITORY_DIR}
os.chdir(REPOSITORY_DIR)
%pip install -q -r requirements.txt

In [ ]:
from src.data.gsm8k import load_gsm8k

DEVELOPMENT_SIZE = 100
RANDOM_SEED = 42
gsm8k = load_gsm8k(development_size=DEVELOPMENT_SIZE, seed=RANDOM_SEED)
print({name: len(split) for name, split in gsm8k.items()})

In [ ]:
EXPECTED_FIELDS = {"id", "question", "reference_solution", "reference_answer"}
assert set(gsm8k["train"].column_names) == EXPECTED_FIELDS
assert set(gsm8k["test"].column_names) == EXPECTED_FIELDS
assert len(gsm8k["development"]) == DEVELOPMENT_SIZE
assert all(row["id"].startswith("gsm8k-train-") for row in gsm8k["development"])
assert all(row["id"].startswith("gsm8k-test-") for row in gsm8k["test"].select(range(5)))
print("Split isolation and normalized schema validated.")

In [ ]:
gsm8k["development"].select(range(3)).to_pandas()

In [ ]:
repeat = load_gsm8k(development_size=DEVELOPMENT_SIZE, seed=RANDOM_SEED)
assert repeat["development"]["id"] == gsm8k["development"]["id"]
print(f"Deterministic sampling confirmed with seed {RANDOM_SEED}.")